In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.utils import resample

from scipy.stats import pearsonr

# Configuration

In [ ]:
target_in = 'FValue'  # 'FValue' (phi_tau) or 'W1' (Delta_tau)

train_path = './training_data/bias_train_e3.csv'
test_path = './training_data/bias_val_e3.csv'

# Load Data and Build Features

In [ ]:
data_train = pd.read_csv(train_path)
data_val = pd.read_csv(test_path)

feats_lin = ['x1', 'x2', 'x3', 'x4', 'x5', 'x6']

def build_features(data):
    df = pd.DataFrame()
    df['x1'] = data['gamma1']
    df['x2'] = data['lambda1']
    df['x3'] = data['delta']
    df['x4'] = data['epsilon']
    df['x5'] = data['NRow']
    df['x6'] = data['NCol']

    for i, fi in enumerate(feats_lin):
        for j in range(i+1, len(feats_lin)):
            df[fi+feats_lin[j]] = df[fi] * df[feats_lin[j]]

    for feat in feats_lin:
        df[feat+feat] = df[feat]**2

    return df

data_train_all = build_features(data_train)
data_val_all = build_features(data_val)

y_train = np.array(data_train[target_in])
y_val = np.array(data_val[target_in])

# Forward Stepwise Feature Selection (10-fold CV)

In [ ]:
def forward_stepwise_regression(X, y, threshold_in=0, verbose=False, k_folds=10):

    selected_features = []
    remaining_features = list(X.columns)

    kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

    score_order = []
    score_order_train = []

    while remaining_features:
        
        best_feature = None
        best_score = float('inf')

        for feature in remaining_features:
            
            features_to_test = selected_features + [feature]
            X_subset = np.array(X[features_to_test])

            split_scores = np.zeros(k_folds)
            split_scores_train = np.zeros(k_folds)

            index = 0

            for train_index, test_index in kf.split(X_subset):

                X_train, X_test = X_subset[train_index], X_subset[test_index]
                y_train_cv, y_test_cv = y[train_index], y[test_index]

                model = LinearRegression().fit(X_train, y_train_cv)

                y_pred_train = model.predict(X_train)
                split_scores_train[index] = mean_squared_error(y_train_cv, y_pred_train) / np.var(y_train_cv)
                
                y_pred = model.predict(X_test)
                split_scores[index] = mean_squared_error(y_test_cv, y_pred) / np.var(y_test_cv)
                
                index += 1

            score_avg = np.average(split_scores)
            score_avg_train = np.average(split_scores_train)

            if score_avg < best_score:
                best_feature = feature
                best_score = score_avg
                best_score_train = score_avg_train

        selected_features.append(best_feature)
        remaining_features.remove(best_feature)
        score_order.append(best_score)
        score_order_train.append(best_score_train)
        
        if verbose:
            print(f'Added feature: {best_feature}, NMSE: {best_score}')
        
        if best_score < threshold_in:
            break

    return selected_features, score_order, score_order_train

In [ ]:
ordered_feats, ordered_scores, ordered_scores_train = forward_stepwise_regression(
    data_train_all, y_train, verbose=True
)

In [ ]:
print('Feature selection order:')
for i, feat in enumerate(ordered_feats):
    print(f'  {i+1}. {feat}')

In [ ]:
np.save(f'./outputs/{target_in}_cv_scores.npy', ordered_scores)
np.save(f'./outputs/{target_in}_train_scores.npy', ordered_scores_train)

# Feature Importance: t-test on Selected Model

In [ ]:
elbow_index = 10

model_feats = ordered_feats[:elbow_index]
print(f'Selected features (first {elbow_index}): {model_feats}')

X_in = data_train_all[model_feats]
X_in = sm.add_constant(X_in)
Y_in = y_train

model = sm.OLS(Y_in, X_in).fit()
print(model.summary())